# Colab’da Ollama Çalıştırma

Bu notebook, Ollama’yı Google Colab içinde çalıştırmayı gösterir.
Küçük–orta ölçekli modelleri ücretsiz kullanabilirsin.

Model listesi: https://ollama.com/library


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Başlamadan Önce

- Runtime → Change runtime type → **T4 GPU seç**
- Runtime → View resources ile kaynakları kontrol et


In [2]:
!sudo apt update
!sudo apt upgrade
!sudo apt install -y pciutils zstd


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,546 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.1 MB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-dri

In [3]:
!curl -fsSL https://ollama.com/install.sh | sh


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## Ollama’yı Başlat


In [14]:
!ollama pull llama3.2:3b-instruct-q8_0

In [25]:
import threading
import subprocess
import time
import os

# 1. Konfiguracja środowiska
os.environ["OLLAMA_MODELS"] = "/content/drive/MyDrive/MODELS"

# 2. Funkcja uruchamiająca serwer Ollama w tle
def run_ollama_serve():
    # stdout i stderr przekierowane do DEVNULL, aby nie blokować konsoli logami serwera
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 3. Start serwera w osobnym wątku
thread = threading.Thread(target=run_ollama_serve, daemon=True)
thread.start()

print("---> Uruchamianie serwera Ollama w tle...")
time.sleep(15)  # Czekamy 15 sekund, aby GPU T4 zdążyło zainicjować serwer
print("---> Serwer powinien być gotowy. Sprawdzam połączenie i uruchamiam skrypt...\n")

# 4. Uruchomienie Twojego skryptu analitycznego
# Używamy os.system, aby widzieć output skryptu w czasie rzeczywistym w Colabie
exit_code = os.system("python /content/oll.py")

if exit_code == 0:
    print("\n[SUKCES] Analiza zakończona pomyślnie.")
else:
    print(f"\n[BŁĄD] Skrypt zakończył się kodem błędu: {exit_code}")

---> Uruchamianie serwera Ollama w tle...
---> Serwer powinien być gotowy. Sprawdzam połączenie i uruchamiam skrypt...


[BŁĄD] Skrypt zakończył się kodem błędu: 256


## Test Et


In [9]:
!curl -s http://127.0.0.1:11434/api/tags


## Model İndir


In [17]:
!ollama pull phi4-mini:3.8b-q8_0 && ollama pull rnj-1:8b


## Model Listele


In [37]:
!ollama list


Error: could not connect to ollama server, run 'ollama serve' to start it


## Kullanım Örneği


In [6]:
!pip install langchain-ollama


In [ ]:
import urllib.request
import json
import os
import re

# ==========================================
# KONFIGURACJA ŚCIEŻEK
# ==========================================
BASE_DIR = "/Volumes/SAMS/GPWDOCKER/A/wyniki_platinum_ideal/20260425_222356/"
INPUT_FILE = os.path.join(BASE_DIR, "top20.json")

# Adresy lokalnego API Ollama
OLLAMA_API_TAGS = "http://localhost:11434/api/tags"
OLLAMA_API_GENERATE = "http://localhost:11434/api/generate"

# ==========================================
# PROMPT ANALITYCZNY
# ==========================================
SYSTEM_PROMPT = """You are a senior equity broker and technical market analyst specializing in short-term and swing-trading evaluation of stocks and ETFs.

You will receive a JSON array. Each array item represents one instrument and usually contains these sections:
- meta
- data_quality
- indicators
- signals
- levels
- ichimoku
- price_action
- patterns
- mtf
- scores
- alerts
- chart

Your task is to read the FULL JSON carefully and produce a broker-style market opinion in clear, practical language.

IMPORTANT GOAL:
Give a selective trading opinion, not a generic summary. Think like a discretionary broker reviewing a watchlist before market action. Identify the best setups, the weakest setups, the hidden risks, and the most actionable candidates.

ANALYSIS RULES

1. Read the entire JSON before making conclusions.
2. Do not blindly trust the field "rating" or "composite_score". Treat them as inputs, not final truth.
3. Give higher weight to signal confluence than to any single indicator.
4. Penalize contradictions strongly. Example:
- bullish score but bearish market_structure,
- strong trend score but DEATH_CROSS_ACTIVE,
- high composite score but weak or negative mtf_alignment,
- bullish daily signals but weekly trend down,
- price near resistance with poor reward/risk.
5. Prefer instruments with:
- good multi-timeframe alignment,
- bullish market structure,
- price above or constructively inside Ichimoku cloud,
- healthy trend regime,
- supportive volume/OBV/AD trends,
- acceptable distance to support and enough upside to resistance,
- reasonable volatility for the setup.
6. Be skeptical of:
- overbought conditions with low upside,
- squeeze/breakout alerts that coexist with bearish structure,
- extreme volume climaxes without follow-through,
- ranging regime with little room to nearest resistance,
- setups where the implied reward/risk is poor.
7. Use the chart arrays only as supporting context, not as the sole basis of the conclusion.
8. If some fields are missing or inconsistent, say so and reduce confidence.
9. Never invent fundamental information, news, earnings, valuation, or macro context unless it is explicitly present in the JSON.
10. This is technical analysis, not investment advice. Write like a broker, but avoid guarantees.

HOW TO INTERPRET THE JSON

Use these areas explicitly:
- meta: ticker, price, daily change, turnover.
- data_quality: trustworthiness of the input.
- indicators: trend, momentum, volatility, volume, support/resistance, structure, pattern and oscillator context.
- signals: moving-average bias.
- levels: pivots, Fibonacci retracements/extensions, nearby technical barriers.
- ichimoku: cloud location and trend confirmation.
- price_action: supports, resistances, distance to barriers, market structure, chart patterns, gaps, trend channel.
- patterns: candlestick pattern context, but treat it as secondary unless confirmed elsewhere.
- mtf: weekly/monthly alignment and trend confirmation.
- scores: trend/momentum/volume/volatility/pattern/composite/confidence/risk-adjusted.
- alerts: critical warnings and opportunity clusters.

WEIGHTING PRIORITIES

Use this approximate decision hierarchy:
1. Multi-timeframe alignment and market structure.
2. Trend regime and price location versus key supports/resistances.
3. Alert quality and contradictions.
4. Volume confirmation.
5. Momentum and oscillators.
6. Candlestick patterns and secondary signals.

REQUIRED OUTPUT STYLE

Write in Polish - ONLY POLISH.
Sound like a professional broker speaking to an informed client.
Be direct, selective, and practical.
Do not use vague phrases like “looks good overall” without reasons.
Every positive or negative opinion must be justified by specific JSON fields.

REQUIRED OUTPUT FORMAT

Use exactly this structure:

# Market View

In 1 short paragraph, describe the overall quality of the watchlist:
- Is the basket broadly bullish, mixed, or fragile?
- Is this a market for aggressive buying, selective buying, breakout entries, or patience?

# Best Opportunities

List the top 4 to 6 instruments.
For each instrument, use this template:

## [TICKER]
- Stance: Accumulate / Buy on pullback / Buy on breakout / Watch / Avoid for now
- Why it stands out: 2-4 bullet points based on JSON evidence
- Main risk: 1-2 bullet points
- Trading logic: 2-3 sentences describing what kind of entry makes sense

# Weakest Setups

List the 3 to 5 weakest or most problematic instruments.
For each instrument, use this template:

## [TICKER]
- Stance: Watch / Avoid for now / Speculative only
- What weakens the setup: 2-4 bullet points
- What would improve it: 1-2 bullet points

# Ranked Table

Create a markdown table with these columns:
| Rank | Ticker | Stance | Technical Quality | Risk Level | Key Reason |

Rules:
- Rank all instruments from strongest to weakest.
- "Technical Quality" must be one of: Excellent / Good / Mixed / Weak
- "Risk Level" must be one of: Low / Medium / High / Very High
- "Key Reason" must be a short phrase, not a paragraph.

# Broker Conclusion

Write 1 concise paragraph:
- Name the final shortlist for action now.
- Name the instruments that should only be monitored.
- Mention whether entries should be made immediately, on pullback, or only after confirmation.

DECISION FRAMEWORK FOR STANCE

Use these interpretations:
- Accumulate: strong confluence, acceptable risk/reward, constructive structure, suitable for scaling in.
- Buy on pullback: bullish setup but entry is extended or near resistance.
- Buy on breakout: setup needs confirmation through resistance or post-squeeze expansion.
- Watch: interesting but incomplete or conflicting setup.
- Avoid for now: negative structure, poor alignment, or bad reward/risk.
- Speculative only: highly volatile and contradictory, suitable only for high-risk traders.

DO NOT
- Do not summarize every ticker equally.
- Do not just repeat indicator values without interpretation.
- Do not overpraise a stock only because scores are high.
- Do not ignore bearish alerts.
- Do not produce investment-advice disclaimers in every section.
- Do not say “insufficient data” unless the JSON really lacks core sections.

QUALITY CHECK BEFORE FINAL ANSWER

Before writing, silently verify:
- Did I read the whole JSON?
- Did I identify contradictions?
- Did I prioritize MTF + structure over raw scores?
- Did I separate the truly actionable setups from merely good-looking ones?
- Did I explain why some “STRONG BUY” names may still be only “Watch” or “Avoid for now”?

Now analyze the provided JSON and generate the final broker report."""

def get_sorted_models():
    """Pobiera listę lokalnych modeli z API Ollamy i sortuje je rosnąco według rozmiaru na dysku."""
    req = urllib.request.Request(OLLAMA_API_TAGS)
    try:
        with urllib.request.urlopen(req) as response:
            data = json.loads(response.read().decode())
    except Exception as e:
        print(f"Błąd połączenia z Ollama. Upewnij się, że aplikacja jest uruchomiona. Szczegóły: {e}")
        return []

    models = data.get("models", [])
    # Sortowanie rosnąco po polu 'size' (w bajtach)
    models.sort(key=lambda x: x.get("size", 0))
    # Zwraca tylko nazwy modeli
    return [m["name"] for m in models]

def remove_thinking_tags(text):
    """Usuwa wewnętrzne 'rozmyślania' modeli (np. sekcje <think> z modeli klasy reasoning)."""
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

def run_model(model_name, prompt, json_data):
    """Wysyła zapytanie do konkretnego modelu Ollamy."""
    print(f"\n---> Ładowanie i analiza modelu: {model_name} (To może chwilę potrwać...)")

    full_prompt = f"{prompt}\n\nHere is the JSON data to analyze:\n{json.dumps(json_data)}"

    payload = {
        "model": model_name,
        "prompt": full_prompt,
        "stream": False,
        "options": {
            "temperature": 0.1,    # Bardzo niska temperatura dla ograniczenia halucynacji
            "num_ctx": 16384       # Duże okno kontekstowe, zapobiega ucinaniu JSON-a
        }
    }

    data = json.dumps(payload).encode('utf-8')
    req = urllib.request.Request(OLLAMA_API_GENERATE, data=data, headers={'Content-Type': 'application/json'})

    try:
        with urllib.request.urlopen(req) as response:
            result = json.loads(response.read().decode())
            raw_response = result.get("response", "")
            return remove_thinking_tags(raw_response)
    except Exception as e:
        print(f"Błąd generowania odpowiedzi dla {model_name}: {e}")
        return None

def unload_model(model_name):
    """Natychmiastowo wyładowuje model z pamięci RAM/VRAM ustawiając keep_alive na 0."""
    print(f"---> Wyładowywanie modelu z pamięci: {model_name}...")

    payload = {
        "model": model_name,
        "keep_alive": 0
    }

    data = json.dumps(payload).encode('utf-8')
    req = urllib.request.Request(OLLAMA_API_GENERATE, data=data, headers={'Content-Type': 'application/json'})

    try:
        with urllib.request.urlopen(req) as response:
            # Sukces - odpowiedź API oznacza wyładowanie z pamięci
            print(f"     [OK] Model {model_name} pomyślnie wyładowany z pamięci.")
    except Exception as e:
        print(f"     [BŁĄD] Nie udało się wyładować modelu {model_name}. Szczegóły: {e}")

def main():
    # 1. Weryfikacja pliku wejściowego
    if not os.path.exists(INPUT_FILE):
        print(f"BŁĄD: Nie znaleziono pliku wejściowego pod ścieżką:\n{INPUT_FILE}")
        return

    print(f"Wczytywanie danych wejściowych z: {INPUT_FILE}")
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        try:
            json_data = json.load(f)
        except json.JSONDecodeError as e:
            print(f"BŁĄD: Plik nie jest poprawnym plikiem JSON: {e}")
            return

    # 2. Pobranie i posortowanie listy modeli
    models = get_sorted_models()
    if not models:
        return

    print(f"\nZnaleziono {len(models)} lokalnych modeli. Kolejność wykonania (od najmniejszego do największego):")
    for idx, m in enumerate(models, 1):
        print(f"{idx}. {m}")

    # 3. Pętla przetwarzająca wszystkie modele sekwencyjnie
    for model in models:
        # A. Uruchomienie analizy na modelu
        output_markdown = run_model(model, SYSTEM_PROMPT, json_data)

        # B. Zapisanie wyników, jeśli analiza się powiodła
        if output_markdown:
            # Zamiana znaków specjalnych (jak dwukropek, np. w nazwie 'llama3:8b') na podkreślniki w nazwie pliku
            safe_model_name = model.replace(":", "_").replace("/", "_")
            output_file = os.path.join(BASE_DIR, f"{safe_model_name}.md")

            with open(output_file, 'w', encoding='utf-8') as f:
                f.write(output_markdown)
            print(f"     ZAKOŃCZONO ANALIZĘ. Raport zapisano jako: {output_file}")

        # C. Natychmiastowe zwalnianie zunifikowanej pamięci (zanim pętla przejdzie do następnego modelu)
        unload_model(model)

if __name__ == "__main__":
    main()


In [22]:
!python3 sync_models.py

=== SYNCHRONIZACJA I PORZĄDKOWANIE MODELI ===

Stopping Ollama processes...
Found files in local Colab: /root/.ollama/models
Moving manifests and metadata to GDrive...
Cleaning up local Colab storage...
[OK] Local storage cleaned.
[OK] Manifests are now on GDrive.

Final GDrive size (/content/drive/MyDrive/MODELS):
66G	/content/drive/MyDrive/MODELS

GOTOWE. Teraz uruchom serwer ponownie, upewniając się, że OLLAMA_MODELS jest ustawiona.


In [33]:
import urllib.request
import json
import os
import re
import subprocess
import time

# ==========================================
# KONFIGURACJA ŚCIEŻEK (Google Colab)
# ==========================================
BASE_DIR = "/content/reports"
INPUT_FILE = "/content/top20.json"
MODELS_DIR = "/content/drive/MyDrive/MODELS"

# ==========================================
# KONFIGURACJA OLLAMA
# ==========================================
os.environ["OLLAMA_MODELS"] = MODELS_DIR

OLLAMA_API_TAGS = "http://localhost:11434/api/tags"
OLLAMA_API_GENERATE = "http://localhost:11434/api/generate"

# Parametry modelu
CONTEXT_WINDOW = 16384
TEMPERATURE = 0.1
KEEP_ALIVE = 0

# ==========================================
# PROMPT ANALITYCZNY
# ==========================================
SYSTEM_PROMPT = """You are a senior equity broker and technical market analyst specializing in short-term and swing-trading evaluation of stocks and ETFs.

You will receive a JSON array. Each array item represents one instrument and usually contains these sections:
- meta
- data_quality
- indicators
- signals
- levels
- ichimoku
- price_action
- patterns
- mtf
- scores
- alerts
- chart

Your task is to read the FULL JSON carefully and produce a broker-style market opinion in clear, practical language.

IMPORTANT GOAL:
Give a selective trading opinion, not a generic summary. Think like a discretionary broker reviewing a watchlist before market action. Identify the best setups, the weakest setups, the hidden risks, and the most actionable candidates.

ANALYSIS RULES

1. Read the entire JSON before making conclusions.
2. Do not blindly trust the field "rating" or "composite_score". Treat them as inputs, not final truth.
3. Give higher weight to signal confluence than to any single indicator.
4. Penalize contradictions strongly. Example:
- bullish score but bearish market_structure,
- strong trend score but DEATH_CROSS_ACTIVE,
- high composite score but weak or negative mtf_alignment,
- bullish daily signals but weekly trend down,
- price near resistance with poor reward/risk.
5. Prefer instruments with:
- good multi-timeframe alignment,
- bullish market structure,
- price above or constructively inside Ichimoku cloud,
- healthy trend regime,
- supportive volume/OBV/AD trends,
- acceptable distance to support and enough upside to resistance,
- reasonable volatility for the setup.
6. Be skeptical of:
- overbought conditions with low upside,
- squeeze/breakout alerts that coexist with bearish structure,
- extreme volume climaxes without follow-through,
- ranging regime with little room to nearest resistance,
- setups where the implied reward/risk is poor.
7. Use the chart arrays only as supporting context, not as the sole basis of the conclusion.
8. If some fields are missing or inconsistent, say so and reduce confidence.
9. Never invent fundamental information, news, earnings, valuation, or macro context unless it is explicitly present in the JSON.
10. This is technical analysis, not investment advice. Write like a broker, but avoid guarantees.

HOW TO INTERPRET THE JSON

Use these areas explicitly:
- meta: ticker, price, daily change, turnover.
- data_quality: trustworthiness of the input.
- indicators: trend, momentum, volatility, volume, support/resistance, structure, pattern and oscillator context.
- signals: moving-average bias.
- levels: pivots, Fibonacci retracements/extensions, nearby technical barriers.
- ichimoku: cloud location and trend confirmation.
- price_action: supports, resistances, distance to barriers, market structure, chart patterns, gaps, trend channel.
- patterns: candlestick pattern context, but treat it as secondary unless confirmed elsewhere.
- mtf: weekly/monthly alignment and trend confirmation.
- scores: trend/momentum/volume/volatility/pattern/composite/confidence/risk-adjusted.
- alerts: critical warnings and opportunity clusters.

WEIGHTING PRIORITIES

Use this approximate decision hierarchy:
1. Multi-timeframe alignment and market structure.
2. Trend regime and price location versus key supports/resistances.
3. Alert quality and contradictions.
4. Volume confirmation.
5. Momentum and oscillators.
6. Candlestick patterns and secondary signals.

REQUIRED OUTPUT STYLE

Write in Polish - ONLY POLISH.
Sound like a professional broker speaking to an informed client.
Be direct, selective, and practical.
Do not use vague phrases like “looks good overall” without reasons.
Every positive or negative opinion must be justified by specific JSON fields.

REQUIRED OUTPUT FORMAT

Use exactly this structure:

# Market View

In 1 short paragraph, describe the overall quality of the watchlist:
- Is the basket broadly bullish, mixed, or fragile?
- Is this a market for aggressive buying, selective buying, breakout entries, or patience?

# Best Opportunities

List the top 4 to 6 instruments.
For each instrument, use this template:

## [TICKER]
- Stance: Accumulate / Buy on pullback / Buy on breakout / Watch / Avoid for now
- Why it stands out: 2-4 bullet points based on JSON evidence
- Main risk: 1-2 bullet points
- Trading logic: 2-3 sentences describing what kind of entry makes sense

# Weakest Setups

List the 3 to 5 weakest or most problematic instruments.
For each instrument, use this template:

## [TICKER]
- Stance: Watch / Avoid for now / Speculative only
- What weakens the setup: 2-4 bullet points
- What would improve it: 1-2 bullet points

# Ranked Table

Create a markdown table with these columns:
| Rank | Ticker | Stance | Technical Quality | Risk Level | Key Reason |

Rules:
- Rank all instruments from strongest to weakest.
- "Technical Quality" must be one of: Excellent / Good / Mixed / Weak
- "Risk Level" must be one of: Low / Medium / High / Very High
- "Key Reason" must be a short phrase, not a paragraph.

# Broker Conclusion

Write 1 concise paragraph:
- Name the final shortlist for action now.
- Name the instruments that should only be monitored.
- Mention whether entries should be made immediately, on pullback, or only after confirmation.

DECISION FRAMEWORK FOR STANCE

Use these interpretations:
- Accumulate: strong confluence, acceptable risk/reward, constructive structure, suitable for scaling in.
- Buy on pullback: bullish setup but entry is extended or near resistance.
- Buy on breakout: setup needs confirmation through resistance or post-squeeze expansion.
- Watch: interesting but incomplete or conflicting setup.
- Avoid for now: negative structure, poor alignment, or bad reward/risk.
- Speculative only: highly volatile and contradictory, suitable only for high-risk traders.

DO NOT
- Do not summarize every ticker equally.
- Do not just repeat indicator values without interpretation.
- Do not overpraise a stock only because scores are high.
- Do not ignore bearish alerts.
- Do not produce investment-advice disclaimers in every section.
- Do not say “insufficient data” unless the JSON really lacks core sections.

QUALITY CHECK BEFORE FINAL ANSWER

Before writing, silently verify:
- Did I read the whole JSON?
- Did I identify contradictions?
- Did I prioritize MTF + structure over raw scores?
- Did I separate the truly actionable setups from merely good-looking ones?
- Did I explain why some “STRONG BUY” names may still be only “Watch” or “Avoid for now”?

Now analyze the provided JSON and generate the final broker report."""

def get_gpu_memory():
    """Pobiera aktualne zużycie VRAM GPU za pomocą nvidia-smi."""
    try:
        output = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
            encoding='utf-8'
        )
        return f"{output.strip()} MB"
    except:
        return "N/A"

def is_any_model_loaded():
    """Sprawdza czy jakikolwiek model jest obecnie w pamięci VRAM."""
    try:
        output = subprocess.check_output(["ollama", "ps"], encoding='utf-8')
        lines = output.strip().split('\n')
        return len(lines) > 1
    except:
        return False

def get_sorted_models():
    """Pobiera listę lokalnych modeli z API Ollamy i sortuje je rosnąco według rozmiaru."""
    req = urllib.request.Request(OLLAMA_API_TAGS)
    try:
        with urllib.request.urlopen(req) as response:
            data = json.loads(response.read().decode())
    except Exception as e:
        print(f"Błąd połączenia z Ollama: {e}")
        return []

    models = data.get("models", [])
    models.sort(key=lambda x: x.get("size", 0))
    return [m["name"] for m in models]

def remove_thinking_tags(text):
    """Usuwa tagi <think> z odpowiedzi modeli reasoning."""
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

def run_model(model_name, prompt, json_data):
    """Wysyła zapytanie do konkretnego modelu Ollamy."""
    full_prompt = f"{prompt}\n\nHere is the JSON data to analyze:\n{json.dumps(json_data)}"

    payload = {
        "model": model_name,
        "prompt": full_prompt,
        "stream": False,
        "options": {
            "temperature": TEMPERATURE,
            "num_ctx": CONTEXT_WINDOW
        },
        "keep_alive": 0
    }

    data = json.dumps(payload).encode('utf-8')
    req = urllib.request.Request(OLLAMA_API_GENERATE, data=data, headers={'Content-Type': 'application/json'})

    try:
        with urllib.request.urlopen(req) as response:
            result = json.loads(response.read().decode())
            raw_response = result.get("response", "")
            return remove_thinking_tags(raw_response)
    except Exception as e:
        print(f"\nBłąd generowania odpowiedzi dla {model_name}: {e}")
        return None

def force_unload(model_name):
    """Próbuje wyładować model i czeka aż faktycznie zniknie z GPU."""
    print(f"      -> Zwalnianie VRAM...")
    payload = {"model": model_name, "keep_alive": 0}
    data = json.dumps(payload).encode('utf-8')
    req = urllib.request.Request(OLLAMA_API_GENERATE, data=data, headers={'Content-Type': 'application/json'})
    try:
        with urllib.request.urlopen(req) as response:
            pass
    except:
        pass

    # Czekaj max 30 sekund aż model zniknie z 'ollama ps'
    for _ in range(30):
        if not is_any_model_loaded():
            return True
        time.sleep(1)
    return False

def main():
    if not os.path.exists(BASE_DIR):
        os.makedirs(BASE_DIR)

    if not os.path.exists(INPUT_FILE):
        print(f"BŁĄD: Brak pliku {INPUT_FILE}")
        return

    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        try:
            json_data = json.load(f)
        except Exception as e:
            print(f"BŁĄD JSON: {e}")
            return

    all_models = get_sorted_models()
    if not all_models:
        return

    print("\n=== MENU WYBORU MODELI ===")
    for idx, m in enumerate(all_models, 1):
        print(f"[{idx}] {m}")

    user_input = input("\nWpisz numery modeli (np. 1, 3, 5) lub 'all': ").strip().lower()

    selected_models = []
    if user_input == 'all':
        selected_models = all_models
    else:
        try:
            indices = [int(i.strip()) - 1 for i in user_input.split(',')]
            selected_models = [all_models[i] for i in indices if 0 <= i < len(all_models)]
        except:
            print("Nieprawidłowy wybór.")
            return

    if not selected_models:
        print("Nie wybrano żadnych modeli.")
        return

    total = len(selected_models)
    print(f"\nRozpoczynam pracę nad {total} modelami...\n")

    for i, model in enumerate(selected_models, 1):
        # Bezpieczeństwo: sprawdź czy coś nie zostało w GPU przed startem
        if is_any_model_loaded():
            print("!!! Wykryto inny model w VRAM. Próba czyszczenia...")
            force_unload(model)

        progress = (i / total) * 100
        gpu_usage = get_gpu_memory()

        print(f"[{progress:3.0f}%] Aktywny model: {model} | VRAM: {gpu_usage}")

        output_markdown = run_model(model, SYSTEM_PROMPT, json_data)

        if output_markdown:
            safe_model_name = model.replace(":", "_").replace("/", "_")
            output_file = os.path.join(BASE_DIR, f"{safe_model_name}.md")
            with open(output_file, 'w', encoding='utf-8') as f:
                f.write(output_markdown)
            print(f"      -> Raport zapisany: {output_file}")

        if not force_unload(model):
            print("!!! OSTRZEŻENIE: Model nie zwolnił VRAM. Możliwe zawieszenie procesu.")

        time.sleep(1)

    print(f"\n[100%] Zakończono analizę wszystkich wybranych modeli ({total}).")

if __name__ == "__main__":
    main()


Błąd połączenia z Ollama: <urlopen error [Errno 111] Connection refused>


In [32]:
!python3 kill_stuck.py

!!! WYKRYTO ZAWIESZENIE MODELU !!!
Zabijam wszystkie procesy Ollama...
Restartuję serwer Ollama w tle...
Serwer zrestartowany. Sprawdzam status:
NAME                                    ID              SIZE      MODIFIED       
translatdfdfdegemma:4b-it-ggggq4_K_M    c49d986b0764    3.3 GB    29 minutes ago    
llama3.2:3b-instruct-q8_0               e410b836fe61    3.4 GB    33 minutes ago    
phi4-mini:3.8b-q8_0                     0be8e6979181    4.1 GB    33 minutes ago    
rnj-1:8b                                d20e29ab8d0f    5.1 GB    33 minutes ago    
glm4:9b                                 5b699761eca5    5.5 GB    59 minutes ago    
qwen3.5:4b-q4_k_m                       2a654d98e6fb    3.4 GB    36 hours ago      
gemma4:e2b-it-q4_K_M                    7fbdbf8f5e45    7.2 GB    36 hours ago      
gemma4:e4b-it-q4_K_M                    c6eb396dbd59    9.6 GB    36 hours ago      
phi4:latest                             ac896e5b8b34    9.1 GB    36 hours ago      
phi4-min

In [38]:
!python /content/start_ollama.py

---> Uruchamianie serwera Ollama w tle...
---> Oczekiwanie na odpowiedź API (to może potrwać do 60s)...
.time=2026-04-26T10:45:08.884Z level=INFO source=routes.go:1752 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MODELS:/content/drive/MyDrive/MODELS OLLAMA_MULTIUSER_CACHE:false OLLAMA_NEW_ENGINE:false OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.

In [ ]:
%run /content/start_ollama.py

---> Start serwera Ollama...
---> Wykryto 12 modeli. Rozpoczynam automatyczny proces Pipeline.


[PIPELINE 1/12] Przygotowuję: dolphin3:8b
[ANALIZA] Start skryptu oll.py dla: dolphin3:8b
[CZYSZCZENIE] Usunięto manifest modelu dolphin3:8b. Czekam na kolejny...

[PIPELINE 2/12] Przygotowuję: gemma4:e2b-it-q4_K_M
[ANALIZA] Start skryptu oll.py dla: gemma4:e2b-it-q4_K_M
[CZYSZCZENIE] Usunięto manifest modelu gemma4:e2b-it-q4_K_M. Czekam na kolejny...

[PIPELINE 3/12] Przygotowuję: gemma4:e4b-it-q4_K_M
[ANALIZA] Start skryptu oll.py dla: gemma4:e4b-it-q4_K_M
[CZYSZCZENIE] Usunięto manifest modelu gemma4:e4b-it-q4_K_M. Czekam na kolejny...

[PIPELINE 4/12] Przygotowuję: glm4:9b
[ANALIZA] Start skryptu oll.py dla: glm4:9b
[CZYSZCZENIE] Usunięto manifest modelu glm4:9b. Czekam na kolejny...

[PIPELINE 5/12] Przygotowuję: llama3.2:3b-instruct-q8_0
[ANALIZA] Start skryptu oll.py dla: llama3.2:3b-instruct-q8_0
[CZYSZCZENIE] Usunięto manifest modelu llama3.2:3b-instruct-q8_0. Czekam na kolejny...
